# Kaggriculture: Thomas 2944 candidate


In [ ]:
from pathlib import Path
import base64
import hashlib
import io
import os
import tarfile
import zlib

EXPECTED_INPUTS = {
    "bfee70e9daaebeae0737a880f1df8f1c60d0783c59af620136cc0d28ef482bc7": "thomas",
    "4889137f1adf1f9266b2ad5a6ac6700c40e9be6dfe3654e39db375fb55b71341": "herd_safe",
}
EXPECTED_MAIN_SHA256 = "2714b6eb6c67fea70eb1dfd90fff0e25bbf15ea8c3391fadf890c7ae274bae70"
LICENSE_BYTES = base64.b64decode("CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIEFwYWNoZSBMaWNlbnNlCiAgICAgICAgICAgICAgICAgICAgICAgICAgIFZlcnNpb24gMi4wLCBKYW51YXJ5IDIwMDQKICAgICAgICAgICAgICAgICAgICAgICAgaHR0cDovL3d3dy5hcGFjaGUub3JnL2xpY2Vuc2VzLwoKICAgVEVSTVMgQU5EIENPTkRJVElPTlMgRk9SIFVTRSwgUkVQUk9EVUNUSU9OLCBBTkQgRElTVFJJQlVUSU9OCgogICAxLiBEZWZpbml0aW9ucy4KCiAgICAgICJMaWNlbnNlIiBzaGFsbCBtZWFuIHRoZSB0ZXJtcyBhbmQgY29uZGl0aW9ucyBmb3IgdXNlLCByZXByb2R1Y3Rpb24sCiAgICAgIGFuZCBkaXN0cmlidXRpb24gYXMgZGVmaW5lZCBieSBTZWN0aW9ucyAxIHRocm91Z2ggOSBvZiB0aGlzIGRvY3VtZW50LgoKICAgICAgIkxpY2Vuc29yIiBzaGFsbCBtZWFuIHRoZSBjb3B5cmlnaHQgb3duZXIgb3IgZW50aXR5IGF1dGhvcml6ZWQgYnkKICAgICAgdGhlIGNvcHlyaWdodCBvd25lciB0aGF0IGlzIGdyYW50aW5nIHRoZSBMaWNlbnNlLgoKICAgICAgIkxlZ2FsIEVudGl0eSIgc2hhbGwgbWVhbiB0aGUgdW5pb24gb2YgdGhlIGFjdGluZyBlbnRpdHkgYW5kIGFsbAogICAgICBvdGhlciBlbnRpdGllcyB0aGF0IGNvbnRyb2wsIGFyZSBjb250cm9sbGVkIGJ5LCBvciBhcmUgdW5kZXIgY29tbW9uCiAgICAgIGNvbnRyb2wgd2l0aCB0aGF0IGVudGl0eS4gRm9yIHRoZSBwdXJwb3NlcyBvZiB0aGlzIGRlZmluaXRpb24sCiAgICAgICJjb250cm9sIiBtZWFucyAoaSkgdGhlIHBvd2VyLCBkaXJlY3Qgb3IgaW5kaXJlY3QsIHRvIGNhdXNlIHRoZQogICAgICBkaXJlY3Rpb24gb3IgbWFuYWdlbWVudCBvZiBzdWNoIGVudGl0eSwgd2hldGhlciBieSBjb250cmFjdCBvcgogICAgICBvdGhlcndpc2UsIG9yIChpaSkgb3duZXJzaGlwIG9mIGZpZnR5IHBlcmNlbnQgKDUwJSkgb3IgbW9yZSBvZiB0aGUKICAgICAgb3V0c3RhbmRpbmcgc2hhcmVzLCBvciAoaWlpKSBiZW5lZmljaWFsIG93bmVyc2hpcCBvZiBzdWNoIGVudGl0eS4KCiAgICAgICJZb3UiIChvciAiWW91ciIpIHNoYWxsIG1lYW4gYW4gaW5kaXZpZHVhbCBvciBMZWdhbCBFbnRpdHkKICAgICAgZXhlcmNpc2luZyBwZXJtaXNzaW9ucyBncmFudGVkIGJ5IHRoaXMgTGljZW5zZS4KCiAgICAgICJTb3VyY2UiIGZvcm0gc2hhbGwgbWVhbiB0aGUgcHJlZmVycmVkIGZvcm0gZm9yIG1ha2luZyBtb2RpZmljYXRpb25zLAogICAgICBpbmNsdWRpbmcgYnV0IG5vdCBsaW1pdGVkIHRvIHNvZnR3YXJlIHNvdXJjZSBjb2RlLCBkb2N1bWVudGF0aW9uCiAgICAgIHNvdXJjZSwgYW5kIGNvbmZpZ3VyYXRpb24gZmlsZXMuCgogICAgICAiT2JqZWN0IiBmb3JtIHNoYWxsIG1lYW4gYW55IGZvcm0gcmVzdWx0aW5nIGZyb20gbWVjaGFuaWNhbAogICAgICB0cmFuc2Zvcm1hdGlvbiBvciB0cmFuc2xhdGlvbiBvZiBhIFNvdXJjZSBmb3JtLCBpbmNsdWRpbmcgYnV0CiAgICAgIG5vdCBsaW1pdGVkIHRvIGNvbXBpbGVkIG9iamVjdCBjb2RlLCBnZW5lcmF0ZWQgZG9jdW1lbnRhdGlvbiwKICAgICAgYW5kIGNvbnZlcnNpb25zIHRvIG90aGVyIG1lZGlhIHR5cGVzLgoKICAgICAgIldvcmsiIHNoYWxsIG1lYW4gdGhlIHdvcmsgb2YgYXV0aG9yc2hpcCwgd2hldGhlciBpbiBTb3VyY2Ugb3IKICAgICAgT2JqZWN0IGZvcm0sIG1hZGUgYXZhaWxhYmxlIHVuZGVyIHRoZSBMaWNlbnNlLCBhcyBpbmRpY2F0ZWQgYnkgYQogICAgICBjb3B5cmlnaHQgbm90aWNlIHRoYXQgaXMgaW5jbHVkZWQgaW4gb3IgYXR0YWNoZWQgdG8gdGhlIHdvcmsKICAgICAgKGFuIGV4YW1wbGUgaXMgcHJvdmlkZWQgaW4gdGhlIEFwcGVuZGl4IGJlbG93KS4KCiAgICAgICJEZXJpdmF0aXZlIFdvcmtzIiBzaGFsbCBtZWFuIGFueSB3b3JrLCB3aGV0aGVyIGluIFNvdXJjZSBvciBPYmplY3QKICAgICAgZm9ybSwgdGhhdCBpcyBiYXNlZCBvbiAob3IgZGVyaXZlZCBmcm9tKSB0aGUgV29yayBhbmQgZm9yIHdoaWNoIHRoZQogICAgICBlZGl0b3JpYWwgcmV2aXNpb25zLCBhbm5vdGF0aW9ucywgZWxhYm9yYXRpb25zLCBvciBvdGhlciBtb2RpZmljYXRpb25zCiAgICAgIHJlcHJlc2VudCwgYXMgYSB3aG9sZSwgYW4gb3JpZ2luYWwgd29yayBvZiBhdXRob3JzaGlwLiBGb3IgdGhlIHB1cnBvc2VzCiAgICAgIG9mIHRoaXMgTGljZW5zZSwgRGVyaXZhdGl2ZSBXb3JrcyBzaGFsbCBub3QgaW5jbHVkZSB3b3JrcyB0aGF0IHJlbWFpbgogICAgICBzZXBhcmFibGUgZnJvbSwgb3IgbWVyZWx5IGxpbmsgKG9yIGJpbmQgYnkgbmFtZSkgdG8gdGhlIGludGVyZmFjZXMgb2YsCiAgICAgIHRoZSBXb3JrIGFuZCBEZXJpdmF0aXZlIFdvcmtzIHRoZXJlb2YuCgogICAgICAiQ29udHJpYnV0aW9uIiBzaGFsbCBtZWFuIGFueSB3b3JrIG9mIGF1dGhvcnNoaXAsIGluY2x1ZGluZwogICAgICB0aGUgb3JpZ2luYWwgdmVyc2lvbiBvZiB0aGUgV29yayBhbmQgYW55IG1vZGlmaWNhdGlvbnMgb3IgYWRkaXRpb25zCiAgICAgIHRvIHRoYXQgV29yayBvciBEZXJpdmF0aXZlIFdvcmtzIHRoZXJlb2YsIHRoYXQgaXMgaW50ZW50aW9uYWxseQogICAgICBzdWJtaXR0ZWQgdG8gTGljZW5zb3IgZm9yIGluY2x1c2lvbiBpbiB0aGUgV29yayBieSB0aGUgY29weXJpZ2h0IG93bmVyCiAgICAgIG9yIGJ5IGFuIGluZGl2aWR1YWwgb3IgTGVnYWwgRW50aXR5IGF1dGhvcml6ZWQgdG8gc3VibWl0IG9uIGJlaGFsZiBvZgogICAgICB0aGUgY29weXJpZ2h0IG93bmVyLiBGb3IgdGhlIHB1cnBvc2VzIG9mIHRoaXMgZGVmaW5pdGlvbiwgInN1Ym1pdHRlZCIKICAgICAgbWVhbnMgYW55IGZvcm0gb2YgZWxlY3Ryb25pYywgdmVyYmFsLCBvciB3cml0dGVuIGNvbW11bmljYXRpb24gc2VudAogICAgICB0byB0aGUgTGljZW5zb3Igb3IgaXRzIHJlcHJlc2VudGF0aXZlcywgaW5jbHVkaW5nIGJ1dCBub3QgbGltaXRlZCB0bwogICAgICBjb21tdW5pY2F0aW9uIG9uIGVsZWN0cm9uaWMgbWFpbGluZyBsaXN0cywgc291cmNlIGNvZGUgY29udHJvbCBzeXN0ZW1zLAogICAgICBhbmQgaXNzdWUgdHJhY2tpbmcgc3lzdGVtcyB0aGF0IGFyZSBtYW5hZ2VkIGJ5LCBvciBvbiBiZWhhbGYgb2YsIHRoZQogICAgICBMaWNlbnNvciBmb3IgdGhlIHB1cnBvc2Ugb2YgZGlzY3Vzc2luZyBhbmQgaW1wcm92aW5nIHRoZSBXb3JrLCBidXQKICAgICAgZXhjbHVkaW5nIGNvbW11bmljYXRpb24gdGhhdCBpcyBjb25zcGljdW91c2x5IG1hcmtlZCBvciBvdGhlcndpc2UKICAgICAgZGVzaWduYXRlZCBpbiB3cml0aW5nIGJ5IHRoZSBjb3B5cmlnaHQgb3duZXIgYXMgIk5vdCBhIENvbnRyaWJ1dGlvbi4iCgogICAgICAiQ29udHJpYnV0b3IiIHNoYWxsIG1lYW4gTGljZW5zb3IgYW5kIGFueSBpbmRpdmlkdWFsIG9yIExlZ2FsIEVudGl0eQogICAgICBvbiBiZWhhbGYgb2Ygd2hvbSBhIENvbnRyaWJ1dGlvbiBoYXMgYmVlbiByZWNlaXZlZCBieSBMaWNlbnNvciBhbmQKICAgICAgc3Vic2VxdWVudGx5IGluY29ycG9yYXRlZCB3aXRoaW4gdGhlIFdvcmsuCgogICAyLiBHcmFudCBvZiBDb3B5cmlnaHQgTGljZW5zZS4gU3ViamVjdCB0byB0aGUgdGVybXMgYW5kIGNvbmRpdGlvbnMgb2YKICAgICAgdGhpcyBMaWNlbnNlLCBlYWNoIENvbnRyaWJ1dG9yIGhlcmVieSBncmFudHMgdG8gWW91IGEgcGVycGV0dWFsLAogICAgICB3b3JsZHdpZGUsIG5vbi1leGNsdXNpdmUsIG5vLWNoYXJnZSwgcm95YWx0eS1mcmVlLCBpcnJldm9jYWJsZQogICAgICBjb3B5cmlnaHQgbGljZW5zZSB0byByZXByb2R1Y2UsIHByZXBhcmUgRGVyaXZhdGl2ZSBXb3JrcyBvZiwKICAgICAgcHVibGljbHkgZGlzcGxheSwgcHVibGljbHkgcGVyZm9ybSwgc3VibGljZW5zZSwgYW5kIGRpc3RyaWJ1dGUgdGhlCiAgICAgIFdvcmsgYW5kIHN1Y2ggRGVyaXZhdGl2ZSBXb3JrcyBpbiBTb3VyY2Ugb3IgT2JqZWN0IGZvcm0uCgogICAzLiBHcmFudCBvZiBQYXRlbnQgTGljZW5zZS4gU3ViamVjdCB0byB0aGUgdGVybXMgYW5kIGNvbmRpdGlvbnMgb2YKICAgICAgdGhpcyBMaWNlbnNlLCBlYWNoIENvbnRyaWJ1dG9yIGhlcmVieSBncmFudHMgdG8gWW91IGEgcGVycGV0dWFsLAogICAgICB3b3JsZHdpZGUsIG5vbi1leGNsdXNpdmUsIG5vLWNoYXJnZSwgcm95YWx0eS1mcmVlLCBpcnJldm9jYWJsZQogICAgICAoZXhjZXB0IGFzIHN0YXRlZCBpbiB0aGlzIHNlY3Rpb24pIHBhdGVudCBsaWNlbnNlIHRvIG1ha2UsIGhhdmUgbWFkZSwKICAgICAgdXNlLCBvZmZlciB0byBzZWxsLCBzZWxsLCBpbXBvcnQsIGFuZCBvdGhlcndpc2UgdHJhbnNmZXIgdGhlIFdvcmssCiAgICAgIHdoZXJlIHN1Y2ggbGljZW5zZSBhcHBsaWVzIG9ubHkgdG8gdGhvc2UgcGF0ZW50IGNsYWltcyBsaWNlbnNhYmxlCiAgICAgIGJ5IHN1Y2ggQ29udHJpYnV0b3IgdGhhdCBhcmUgbmVjZXNzYXJpbHkgaW5mcmluZ2VkIGJ5IHRoZWlyCiAgICAgIENvbnRyaWJ1dGlvbihzKSBhbG9uZSBvciBieSBjb21iaW5hdGlvbiBvZiB0aGVpciBDb250cmlidXRpb24ocykKICAgICAgd2l0aCB0aGUgV29yayB0byB3aGljaCBzdWNoIENvbnRyaWJ1dGlvbihzKSB3YXMgc3VibWl0dGVkLiBJZiBZb3UKICAgICAgaW5zdGl0dXRlIHBhdGVudCBsaXRpZ2F0aW9uIGFnYWluc3QgYW55IGVudGl0eSAoaW5jbHVkaW5nIGEKICAgICAgY3Jvc3MtY2xhaW0gb3IgY291bnRlcmNsYWltIGluIGEgbGF3c3VpdCkgYWxsZWdpbmcgdGhhdCB0aGUgV29yawogICAgICBvciBhIENvbnRyaWJ1dGlvbiBpbmNvcnBvcmF0ZWQgd2l0aGluIHRoZSBXb3JrIGNvbnN0aXR1dGVzIGRpcmVjdAogICAgICBvciBjb250cmlidXRvcnkgcGF0ZW50IGluZnJpbmdlbWVudCwgdGhlbiBhbnkgcGF0ZW50IGxpY2Vuc2VzCiAgICAgIGdyYW50ZWQgdG8gWW91IHVuZGVyIHRoaXMgTGljZW5zZSBmb3IgdGhhdCBXb3JrIHNoYWxsIHRlcm1pbmF0ZQogICAgICBhcyBvZiB0aGUgZGF0ZSBzdWNoIGxpdGlnYXRpb24gaXMgZmlsZWQuCgogICA0LiBSZWRpc3RyaWJ1dGlvbi4gWW91IG1heSByZXByb2R1Y2UgYW5kIGRpc3RyaWJ1dGUgY29waWVzIG9mIHRoZQogICAgICBXb3JrIG9yIERlcml2YXRpdmUgV29ya3MgdGhlcmVvZiBpbiBhbnkgbWVkaXVtLCB3aXRoIG9yIHdpdGhvdXQKICAgICAgbW9kaWZpY2F0aW9ucywgYW5kIGluIFNvdXJjZSBvciBPYmplY3QgZm9ybSwgcHJvdmlkZWQgdGhhdCBZb3UKICAgICAgbWVldCB0aGUgZm9sbG93aW5nIGNvbmRpdGlvbnM6CgogICAgICAoYSkgWW91IG11c3QgZ2l2ZSBhbnkgb3RoZXIgcmVjaXBpZW50cyBvZiB0aGUgV29yayBvcgogICAgICAgICAgRGVyaXZhdGl2ZSBXb3JrcyBhIGNvcHkgb2YgdGhpcyBMaWNlbnNlOyBhbmQKCiAgICAgIChiKSBZb3UgbXVzdCBjYXVzZSBhbnkgbW9kaWZpZWQgZmlsZXMgdG8gY2FycnkgcHJvbWluZW50IG5vdGljZXMKICAgICAgICAgIHN0YXRpbmcgdGhhdCBZb3UgY2hhbmdlZCB0aGUgZmlsZXM7IGFuZAoKICAgICAgKGMpIFlvdSBtdXN0IHJldGFpbiwgaW4gdGhlIFNvdXJjZSBmb3JtIG9mIGFueSBEZXJpdmF0aXZlIFdvcmtzCiAgICAgICAgICB0aGF0IFlvdSBkaXN0cmlidXRlLCBhbGwgY29weXJpZ2h0LCBwYXRlbnQsIHRyYWRlbWFyaywgYW5kCiAgICAgICAgICBhdHRyaWJ1dGlvbiBub3RpY2VzIGZyb20gdGhlIFNvdXJjZSBmb3JtIG9mIHRoZSBXb3JrLAogICAgICAgICAgZXhjbHVkaW5nIHRob3NlIG5vdGljZXMgdGhhdCBkbyBub3QgcGVydGFpbiB0byBhbnkgcGFydCBvZgogICAgICAgICAgdGhlIERlcml2YXRpdmUgV29ya3M7IGFuZAoKICAgICAgKGQpIElmIHRoZSBXb3JrIGluY2x1ZGVzIGEgIk5PVElDRSIgdGV4dCBmaWxlIGFzIHBhcnQgb2YgaXRzCiAgICAgICAgICBkaXN0cmlidXRpb24sIHRoZW4gYW55IERlcml2YXRpdmUgV29ya3MgdGhhdCBZb3UgZGlzdHJpYnV0ZSBtdXN0CiAgICAgICAgICBpbmNsdWRlIGEgcmVhZGFibGUgY29weSBvZiB0aGUgYXR0cmlidXRpb24gbm90aWNlcyBjb250YWluZWQKICAgICAgICAgIHdpdGhpbiBzdWNoIE5PVElDRSBmaWxlLCBleGNsdWRpbmcgdGhvc2Ugbm90aWNlcyB0aGF0IGRvIG5vdAogICAgICAgICAgcGVydGFpbiB0byBhbnkgcGFydCBvZiB0aGUgRGVyaXZhdGl2ZSBXb3JrcywgaW4gYXQgbGVhc3Qgb25lCiAgICAgICAgICBvZiB0aGUgZm9sbG93aW5nIHBsYWNlczogd2l0aGluIGEgTk9USUNFIHRleHQgZmlsZSBkaXN0cmlidXRlZAogICAgICAgICAgYXMgcGFydCBvZiB0aGUgRGVyaXZhdGl2ZSBXb3Jrczsgd2l0aGluIHRoZSBTb3VyY2UgZm9ybSBvcgogICAgICAgICAgZG9jdW1lbnRhdGlvbiwgaWYgcHJvdmlkZWQgYWxvbmcgd2l0aCB0aGUgRGVyaXZhdGl2ZSBXb3Jrczsgb3IsCiAgICAgICAgICB3aXRoaW4gYSBkaXNwbGF5IGdlbmVyYXRlZCBieSB0aGUgRGVyaXZhdGl2ZSBXb3JrcywgaWYgYW5kCiAgICAgICAgICB3aGVyZXZlciBzdWNoIHRoaXJkLXBhcnR5IG5vdGljZXMgbm9ybWFsbHkgYXBwZWFyLiBUaGUgY29udGVudHMKICAgICAgICAgIG9mIHRoZSBOT1RJQ0UgZmlsZSBhcmUgZm9yIGluZm9ybWF0aW9uYWwgcHVycG9zZXMgb25seSBhbmQKICAgICAgICAgIGRvIG5vdCBtb2RpZnkgdGhlIExpY2Vuc2UuIFlvdSBtYXkgYWRkIFlvdXIgb3duIGF0dHJpYnV0aW9uCiAgICAgICAgICBub3RpY2VzIHdpdGhpbiBEZXJpdmF0aXZlIFdvcmtzIHRoYXQgWW91IGRpc3RyaWJ1dGUsIGFsb25nc2lkZQogICAgICAgICAgb3IgYXMgYW4gYWRkZW5kdW0gdG8gdGhlIE5PVElDRSB0ZXh0IGZyb20gdGhlIFdvcmssIHByb3ZpZGVkCiAgICAgICAgICB0aGF0IHN1Y2ggYWRkaXRpb25hbCBhdHRyaWJ1dGlvbiBub3RpY2VzIGNhbm5vdCBiZSBjb25zdHJ1ZWQKICAgICAgICAgIGFzIG1vZGlmeWluZyB0aGUgTGljZW5zZS4KCiAgICAgIFlvdSBtYXkgYWRkIFlvdXIgb3duIGNvcHlyaWdodCBzdGF0ZW1lbnQgdG8gWW91ciBtb2RpZmljYXRpb25zIGFuZAogICAgICBtYXkgcHJvdmlkZSBhZGRpdGlvbmFsIG9yIGRpZmZlcmVudCBsaWNlbnNlIHRlcm1zIGFuZCBjb25kaXRpb25zCiAgICAgIGZvciB1c2UsIHJlcHJvZHVjdGlvbiwgb3IgZGlzdHJpYnV0aW9uIG9mIFlvdXIgbW9kaWZpY2F0aW9ucywgb3IKICAgICAgZm9yIGFueSBzdWNoIERlcml2YXRpdmUgV29ya3MgYXMgYSB3aG9sZSwgcHJvdmlkZWQgWW91ciB1c2UsCiAgICAgIHJlcHJvZHVjdGlvbiwgYW5kIGRpc3RyaWJ1dGlvbiBvZiB0aGUgV29yayBvdGhlcndpc2UgY29tcGxpZXMgd2l0aAogICAgICB0aGUgY29uZGl0aW9ucyBzdGF0ZWQgaW4gdGhpcyBMaWNlbnNlLgoKICAgNS4gU3VibWlzc2lvbiBvZiBDb250cmlidXRpb25zLiBVbmxlc3MgWW91IGV4cGxpY2l0bHkgc3RhdGUgb3RoZXJ3aXNlLAogICAgICBhbnkgQ29udHJpYnV0aW9uIGludGVudGlvbmFsbHkgc3VibWl0dGVkIGZvciBpbmNsdXNpb24gaW4gdGhlIFdvcmsKICAgICAgYnkgWW91IHRvIHRoZSBMaWNlbnNvciBzaGFsbCBiZSB1bmRlciB0aGUgdGVybXMgYW5kIGNvbmRpdGlvbnMgb2YKICAgICAgdGhpcyBMaWNlbnNlLCB3aXRob3V0IGFueSBhZGRpdGlvbmFsIHRlcm1zIG9yIGNvbmRpdGlvbnMuCiAgICAgIE5vdHdpdGhzdGFuZGluZyB0aGUgYWJvdmUsIG5vdGhpbmcgaGVyZWluIHNoYWxsIHN1cGVyc2VkZSBvciBtb2RpZnkKICAgICAgdGhlIHRlcm1zIG9mIGFueSBzZXBhcmF0ZSBsaWNlbnNlIGFncmVlbWVudCB5b3UgbWF5IGhhdmUgZXhlY3V0ZWQKICAgICAgd2l0aCBMaWNlbnNvciByZWdhcmRpbmcgc3VjaCBDb250cmlidXRpb25zLgoKICAgNi4gVHJhZGVtYXJrcy4gVGhpcyBMaWNlbnNlIGRvZXMgbm90IGdyYW50IHBlcm1pc3Npb24gdG8gdXNlIHRoZSB0cmFkZQogICAgICBuYW1lcywgdHJhZGVtYXJrcywgc2VydmljZSBtYXJrcywgb3IgcHJvZHVjdCBuYW1lcyBvZiB0aGUgTGljZW5zb3IsCiAgICAgIGV4Y2VwdCBhcyByZXF1aXJlZCBmb3IgcmVhc29uYWJsZSBhbmQgY3VzdG9tYXJ5IHVzZSBpbiBkZXNjcmliaW5nIHRoZQogICAgICBvcmlnaW4gb2YgdGhlIFdvcmsgYW5kIHJlcHJvZHVjaW5nIHRoZSBjb250ZW50IG9mIHRoZSBOT1RJQ0UgZmlsZS4KCiAgIDcuIERpc2NsYWltZXIgb2YgV2FycmFudHkuIFVubGVzcyByZXF1aXJlZCBieSBhcHBsaWNhYmxlIGxhdyBvcgogICAgICBhZ3JlZWQgdG8gaW4gd3JpdGluZywgTGljZW5zb3IgcHJvdmlkZXMgdGhlIFdvcmsgKGFuZCBlYWNoCiAgICAgIENvbnRyaWJ1dG9yIHByb3ZpZGVzIGl0cyBDb250cmlidXRpb25zKSBvbiBhbiAiQVMgSVMiIEJBU0lTLAogICAgICBXSVRIT1VUIFdBUlJBTlRJRVMgT1IgQ09ORElUSU9OUyBPRiBBTlkgS0lORCwgZWl0aGVyIGV4cHJlc3Mgb3IKICAgICAgaW1wbGllZCwgaW5jbHVkaW5nLCB3aXRob3V0IGxpbWl0YXRpb24sIGFueSB3YXJyYW50aWVzIG9yIGNvbmRpdGlvbnMKICAgICAgb2YgVElUTEUsIE5PTi1JTkZSSU5HRU1FTlQsIE1FUkNIQU5UQUJJTElUWSwgb3IgRklUTkVTUyBGT1IgQQogICAgICBQQVJUSUNVTEFSIFBVUlBPU0UuIFlvdSBhcmUgc29sZWx5IHJlc3BvbnNpYmxlIGZvciBkZXRlcm1pbmluZyB0aGUKICAgICAgYXBwcm9wcmlhdGVuZXNzIG9mIHVzaW5nIG9yIHJlZGlzdHJpYnV0aW5nIHRoZSBXb3JrIGFuZCBhc3N1bWUgYW55CiAgICAgIHJpc2tzIGFzc29jaWF0ZWQgd2l0aCBZb3VyIGV4ZXJjaXNlIG9mIHBlcm1pc3Npb25zIHVuZGVyIHRoaXMgTGljZW5zZS4KCiAgIDguIExpbWl0YXRpb24gb2YgTGlhYmlsaXR5LiBJbiBubyBldmVudCBhbmQgdW5kZXIgbm8gbGVnYWwgdGhlb3J5LAogICAgICB3aGV0aGVyIGluIHRvcnQgKGluY2x1ZGluZyBuZWdsaWdlbmNlKSwgY29udHJhY3QsIG9yIG90aGVyd2lzZSwKICAgICAgdW5sZXNzIHJlcXVpcmVkIGJ5IGFwcGxpY2FibGUgbGF3IChzdWNoIGFzIGRlbGliZXJhdGUgYW5kIGdyb3NzbHkKICAgICAgbmVnbGlnZW50IGFjdHMpIG9yIGFncmVlZCB0byBpbiB3cml0aW5nLCBzaGFsbCBhbnkgQ29udHJpYnV0b3IgYmUKICAgICAgbGlhYmxlIHRvIFlvdSBmb3IgZGFtYWdlcywgaW5jbHVkaW5nIGFueSBkaXJlY3QsIGluZGlyZWN0LCBzcGVjaWFsLAogICAgICBpbmNpZGVudGFsLCBvciBjb25zZXF1ZW50aWFsIGRhbWFnZXMgb2YgYW55IGNoYXJhY3RlciBhcmlzaW5nIGFzIGEKICAgICAgcmVzdWx0IG9mIHRoaXMgTGljZW5zZSBvciBvdXQgb2YgdGhlIHVzZSBvciBpbmFiaWxpdHkgdG8gdXNlIHRoZQogICAgICBXb3JrIChpbmNsdWRpbmcgYnV0IG5vdCBsaW1pdGVkIHRvIGRhbWFnZXMgZm9yIGxvc3Mgb2YgZ29vZHdpbGwsCiAgICAgIHdvcmsgc3RvcHBhZ2UsIGNvbXB1dGVyIGZhaWx1cmUgb3IgbWFsZnVuY3Rpb24sIG9yIGFueSBhbmQgYWxsCiAgICAgIG90aGVyIGNvbW1lcmNpYWwgZGFtYWdlcyBvciBsb3NzZXMpLCBldmVuIGlmIHN1Y2ggQ29udHJpYnV0b3IKICAgICAgaGFzIGJlZW4gYWR2aXNlZCBvZiB0aGUgcG9zc2liaWxpdHkgb2Ygc3VjaCBkYW1hZ2VzLgoKICAgOS4gQWNjZXB0aW5nIFdhcnJhbnR5IG9yIEFkZGl0aW9uYWwgTGlhYmlsaXR5LiBXaGlsZSByZWRpc3RyaWJ1dGluZwogICAgICB0aGUgV29yayBvciBEZXJpdmF0aXZlIFdvcmtzIHRoZXJlb2YsIFlvdSBtYXkgY2hvb3NlIHRvIG9mZmVyLAogICAgICBhbmQgY2hhcmdlIGEgZmVlIGZvciwgYWNjZXB0YW5jZSBvZiBzdXBwb3J0LCB3YXJyYW50eSwgaW5kZW1uaXR5LAogICAgICBvciBvdGhlciBsaWFiaWxpdHkgb2JsaWdhdGlvbnMgYW5kL29yIHJpZ2h0cyBjb25zaXN0ZW50IHdpdGggdGhpcwogICAgICBMaWNlbnNlLiBIb3dldmVyLCBpbiBhY2NlcHRpbmcgc3VjaCBvYmxpZ2F0aW9ucywgWW91IG1heSBhY3Qgb25seQogICAgICBvbiBZb3VyIG93biBiZWhhbGYgYW5kIG9uIFlvdXIgc29sZSByZXNwb25zaWJpbGl0eSwgbm90IG9uIGJlaGFsZgogICAgICBvZiBhbnkgb3RoZXIgQ29udHJpYnV0b3IsIGFuZCBvbmx5IGlmIFlvdSBhZ3JlZSB0byBpbmRlbW5pZnksCiAgICAgIGRlZmVuZCwgYW5kIGhvbGQgZWFjaCBDb250cmlidXRvciBoYXJtbGVzcyBmb3IgYW55IGxpYWJpbGl0eQogICAgICBpbmN1cnJlZCBieSwgb3IgY2xhaW1zIGFzc2VydGVkIGFnYWluc3QsIHN1Y2ggQ29udHJpYnV0b3IgYnkgcmVhc29uCiAgICAgIG9mIHlvdXIgYWNjZXB0aW5nIGFueSBzdWNoIHdhcnJhbnR5IG9yIGFkZGl0aW9uYWwgbGlhYmlsaXR5LgoKICAgRU5EIE9GIFRFUk1TIEFORCBDT05ESVRJT05TCgogICBBUFBFTkRJWDogSG93IHRvIGFwcGx5IHRoZSBBcGFjaGUgTGljZW5zZSB0byB5b3VyIHdvcmsuCgogICAgICBUbyBhcHBseSB0aGUgQXBhY2hlIExpY2Vuc2UgdG8geW91ciB3b3JrLCBhdHRhY2ggdGhlIGZvbGxvd2luZwogICAgICBib2lsZXJwbGF0ZSBub3RpY2UsIHdpdGggdGhlIGZpZWxkcyBlbmNsb3NlZCBieSBicmFja2V0cyAiW10iCiAgICAgIHJlcGxhY2VkIHdpdGggeW91ciBvd24gaWRlbnRpZnlpbmcgaW5mb3JtYXRpb24uIChEb24ndCBpbmNsdWRlCiAgICAgIHRoZSBicmFja2V0cyEpICBUaGUgdGV4dCBzaG91bGQgYmUgZW5jbG9zZWQgaW4gdGhlIGFwcHJvcHJpYXRlCiAgICAgIGNvbW1lbnQgc3ludGF4IGZvciB0aGUgZmlsZSBmb3JtYXQuIFdlIGFsc28gcmVjb21tZW5kIHRoYXQgYQogICAgICBmaWxlIG9yIGNsYXNzIG5hbWUgYW5kIGRlc2NyaXB0aW9uIG9mIHB1cnBvc2UgYmUgaW5jbHVkZWQgb24gdGhlCiAgICAgIHNhbWUgInByaW50ZWQgcGFnZSIgYXMgdGhlIGNvcHlyaWdodCBub3RpY2UgZm9yIGVhc2llcgogICAgICBpZGVudGlmaWNhdGlvbiB3aXRoaW4gdGhpcmQtcGFydHkgYXJjaGl2ZXMuCgogICBDb3B5cmlnaHQgW3l5eXldIFtuYW1lIG9mIGNvcHlyaWdodCBvd25lcl0KCiAgIExpY2Vuc2VkIHVuZGVyIHRoZSBBcGFjaGUgTGljZW5zZSwgVmVyc2lvbiAyLjAgKHRoZSAiTGljZW5zZSIpOwogICB5b3UgbWF5IG5vdCB1c2UgdGhpcyBmaWxlIGV4Y2VwdCBpbiBjb21wbGlhbmNlIHdpdGggdGhlIExpY2Vuc2UuCiAgIFlvdSBtYXkgb2J0YWluIGEgY29weSBvZiB0aGUgTGljZW5zZSBhdAoKICAgICAgIGh0dHA6Ly93d3cuYXBhY2hlLm9yZy9saWNlbnNlcy9MSUNFTlNFLTIuMAoKICAgVW5sZXNzIHJlcXVpcmVkIGJ5IGFwcGxpY2FibGUgbGF3IG9yIGFncmVlZCB0byBpbiB3cml0aW5nLCBzb2Z0d2FyZQogICBkaXN0cmlidXRlZCB1bmRlciB0aGUgTGljZW5zZSBpcyBkaXN0cmlidXRlZCBvbiBhbiAiQVMgSVMiIEJBU0lTLAogICBXSVRIT1VUIFdBUlJBTlRJRVMgT1IgQ09ORElUSU9OUyBPRiBBTlkgS0lORCwgZWl0aGVyIGV4cHJlc3Mgb3IgaW1wbGllZC4KICAgU2VlIHRoZSBMaWNlbnNlIGZvciB0aGUgc3BlY2lmaWMgbGFuZ3VhZ2UgZ292ZXJuaW5nIHBlcm1pc3Npb25zIGFuZAogICBsaW1pdGF0aW9ucyB1bmRlciB0aGUgTGljZW5zZS4K")
NOTICE_BYTES = base64.b64decode("UmVzY3VlIGNvbnRyb2xsZXIgcmVmcmVzaCwgU2VwdGVtYmVyIDIyLCAyMDI2LiBBcGFjaGUtMi4wLgpEaXJlY3QgYmFzZTogc2hpaWluOSwgaHR0cHM6Ly93d3cua2FnZ2xlLmNvbS9jb2RlL3NoaWlpbjkveW91ci1tYXJrZXQtbGlzdC1pcy1hbi1vcmRlci1ib29rClY1NSBjb250cm9sbGVyIGFuZCBWNTYgaW5wdXQgYnVkZ2V0aW5nOiBBaG1lZCBCZXJhdCBPemVyLgpodHRwczovL3d3dy5rYWdnbGUuY29tL2NvZGUvYWhtZWRiZXJhdG96ZXIva2FnZ3JpY3VsdHVyZS12NTYtc21hcnRlci1zZWVkcy1hbmQtZmVydGlsaXplcgpNZXRhdjQ6IFRob21hcyBUc2NoaW5rZWwuIEhheWFzaGk6IG9yaWdpbmFsIFNob3BSb3V0ZXIuCkRtaXRyaWkgR2x1emRvdjogc2V2ZW4tdHVybiByZXNjdWUsIGFkYXB0YXRpb24sIGV2YWx1YXRpb24gYW5kIGludGVncmF0aW9uLgpBbGwgdXBzdHJlYW0gc291cmNlIG5vdGljZXMgcmVtYWluIGluIG1haW4ucHkuCgpIaXN0b3JpY2FsIFJlc2N1ZSBhdHRyaWJ1dGlvbiBmb2xsb3dzOyBmaWxlLWxheW91dCBzdGF0ZW1lbnRzIGRlc2NyaWJlIHRoZSBvcmlnaW5hbCBwYWNrYWdlLCBub3QgdGhpcyBzaW5nbGUtZmlsZSByZWZyZXNoLgpFMTgyIFNob3AwOTA5ICsgdGVybWluYWwgcGh5c2ljYWwgY2xvc3VyZSAobW9kaWZpZWQgMjAyNi0wOS0wOSkKClRoZSBhY3RpdmUgcHVibGljIHBhcmVudCBpcyBZdXN1a2UgSGF5YXNoaSdzIHloYXk4MS9zaG9wLXJvdXRlci0wOTA5IHYzLgpyb3V0ZXJfcGFyZW50LnB5IGFuZCBhY3Rpb25zLmpzb24gYXJlIGV4YWN0IG9yaWdpbmFsIGJ5dGVzLCBub3QgbmV3bHkgYXV0aG9yZWQKcm91dGVzLiBUaGUgcGFyZW50IGNyZWRpdHMgYXVyYXg3J3MgUmVhY3RpdmUgUm91dGVyIGZvciBzYWxlIHRpbWluZyBhbmQgc2hlZApwcm9qZWN0aW9uOyB0aGF0IGF0dHJpYnV0aW9uIHJlbWFpbnMgaW4gcm91dGVyX3BhcmVudC5weS4gT3JpZ2luYWwgcGF5bG9hZApMSUNFTlNFLnR4dCBpcyBwcmVzZXJ2ZWQgdW5jaGFuZ2VkIChBcGFjaGUgTGljZW5zZSAyLjAgdGV4dCk7IGl0IGNvbnRhaW5zIG5vCm5hbWVkIGNvcHlyaWdodCBncmFudG9yIGFuZCBubyBzZXBhcmF0ZSBOT1RJQ0Ugd2FzIHN1cHBsaWVkLiBObyBhZGRpdGlvbmFsCm93bmVyc2hpcCwgZW5kb3JzZW1lbnQsIG9yIHVwc3RyZWFtIHJlcGxheS1kYXRhIHJpZ2h0cyBjbGFpbSBpcyBtYWRlLgoKTG9jYWwgY2hhbmdlczogc2VwYXJhdGUgbWFpbi5weS9wb2xpY3kucHkgYWRhcHRlcjsgYm91bmRlZCBzdGFydDcxMiBwbGFubmVyCmNvcGllZCBmcm9tIGZyb3plbiBFMTgwL1M3OCBhbmQgbW9kaWZpZWQgZm9yIHNldmVuIGNhbGxiYWNrcywgZXhhY3QgU2hvcCBmaW5hbApsaXF1aWRhdGlvbiwgc3RyaWN0IHBvc2l0aXZlIHBoeXNpY2FsIGRlbGl2ZXJ5L3NhbGUgZ2FpbiwgYW5kIG9ic2VydmF0aW9uIGd1YXJkcy4KdW5pdF9tb2RlbC5weSBpcyBhbiB1bmNoYW5nZWQgZnJvemVuIEUxODAgY29weSBvZiBLYWdnbGUncyBleHRyYWN0ZWQgc2VtYW50aWNzLgpUaGUgZm9sbG93aW5nIG9yaWdpbmFsIEUxODAgbm90aWNlIGlzIHJldGFpbmVkIHZlcmJhdGltIGZvciBhdHRyaWJ1dGlvbiBoaXN0b3J5LgpJdHMgcmVmZXJlbmNlcyB0byBUaG9tYXMgZmlsZXMgZGVzY3JpYmUgRTE4MCwgbm90IGZpbGVzIHN1cHBsaWVkIGluIHRoaXMgU2hvcApwYWNrYWdlOiBubyBUaG9tYXMgdGFwZXMsIHRyZWVzIG9yIHBvbGljeSBhcmUgaW5jbHVkZWQgaGVyZS4KCi0tLS0tIE9yaWdpbmFsIEUxODAgbm90aWNlIC0tLS0tCkthZ2dyaWN1bHR1cmU6IExhc3QtTWlsZSBIYXJ2ZXN0IFBsYW5uZXIKQXR0cmlidXRpb24gYW5kIGNoYW5nZSBub3RpY2UKClRob21hcyBUc2NoaW5rZWwgaXMgdGhlIGF1dGhvciBvZiB0aGUgcGFyZW50IHB1YmxpYyBzdGF0ZS1yb3V0ZXIgcG9saWN5IGFuZCBpdHMKcHVibGlzaGVkIGRlY2lzaW9uIHRyZWVzIGFuZCBhY3Rpb24tcm91dGUgZGF0YS4gU291cmNlOiBLYWdncmljdWx0dXJlOiA5My44JSBXaW4KUmF0ZSBQdWJsaWMgU3RhdGUgUm91dGVyLCBub3RlYm9vayB2ZXJzaW9uIDMsIHNjcmlwdFZlcnNpb25JZCAzNDc5MzYxODM6Cmh0dHBzOi8vd3d3LmthZ2dsZS5jb20vY29kZS90aG9tYXN0c2NoaW5rZWwva2FnZ3JpY3VsdHVyZS05My04LXdpbi1yYXRlLXB1YmxpYy1zdGF0ZS1yb3V0ZXI/c2NyaXB0VmVyc2lvbklkPTM0NzkzNjE4MwpUaGUgcHVibGljIG5vdGVib29rIGlkZW50aWZpZXMgaXRzIGxpY2Vuc2UgYXMgQXBhY2hlIExpY2Vuc2UsIFZlcnNpb24gMi4wLgpPcmlnaW5hbCBwdWJsaXNoZWQgbWFpbi5weSBTSEEtMjU2OgpiODdhMjdlZDYxNGEzMzMyOWJlODVmMWI2NjJlNTFjZjQwNzhhMDE5ZmVlOTM3YWZjZWJiYmJmMmY1MWY4NTIyCgpDaGFuZ2VzIHRvIHRoYXQgc291cmNlIGZvciB0aGlzIGRpc3RyaWJ1dGlvbjogY29tcHJlc3NlZCByb3V0ZS90cmVlIGxpdGVyYWxzCndlcmUgZGVjb2RlZCBpbnRvIHJlYWRhYmxlIHRhcGVzLmpzb24gYW5kIHRyZWVzLmpzb247IGEgcmVhZC1vbmx5IHBsYW5uZWRfYWN0aW9uCmhlbHBlciB3YXMgYWRkZWQ7IGRlc2NyaXB0aXZlIGhlYWRlcnMgYW5kIGxvY2FsIGRhdGEgbG9hZGluZyB3ZXJlIGFkYXB0ZWQuClRoZSBvcmlnaW5hbCBwYXJlbnQgZmVhdHVyZSBleHRyYWN0aW9uLCB0cmVlIHRyYXZlcnNhbCBhbmQgYWdlbnQgYmVoYXZpb3IgYXJlCnJldGFpbmVkLiBUaGVzZSBwdWJsaWMgcm91dGVzIGFyZSBub3QgY2xhaW1lZCBhcyBuZXdseSBhdXRob3JlZCBvciB0cmFpbmVkIGJ5CnRoZSBub3RlYm9vayBkaXN0cmlidXRvci4KCnVuaXRfbW9kZWwucHkgY29udGFpbnMgZGV0ZXJtaW5pc3RpYyB1bml0LWFjdGlvbiBhbmQgY3JvcC1kZWNheSBkZWZpbml0aW9ucwpleHRyYWN0ZWQgZnJvbSBLYWdnbGUncyBrYWdnbGUtZW52aXJvbm1lbnRzIDEuMzIuNyBLYWdncmljdWx0dXJlIGVuZ2luZSwgbGljZW5zZWQKdW5kZXIgQXBhY2hlIExpY2Vuc2UsIFZlcnNpb24gMi4wLiBDcmVkaXQ6IEthZ2dsZSBhbmQgdGhlIGthZ2dsZS1lbnZpcm9ubWVudHMKY29udHJpYnV0b3JzLiBQcm9qZWN0OiBodHRwczovL2dpdGh1Yi5jb20vS2FnZ2xlL2thZ2dsZS1lbnZpcm9ubWVudHMKU291cmNlIGZpbGU6IGthZ2dsZV9lbnZpcm9ubWVudHMvZW52cy9rYWdncmljdWx0dXJlL2thZ2dyaWN1bHR1cmUucHkKU291cmNlIFNIQS0yNTY6CmJjOGE1NDg3OWVmMDJjN2VhNjRiOGIzMzNkNmE5NzZmMGVhNjVjNDk0OTE0OWQwMWY0NjNmMjNiY2NlZTY1M2UKVGhlIGV4dHJhY3RlZCB1bml0L2RlY2F5IGRlZmluaXRpb25zIGFyZSBub3QgYSBuZXdseSBhdXRob3JlZCBnYW1lIGVuZ2luZTsKbWFya2V0IHByaWNlIGR5bmFtaWNzIGFuZCB0aGUgZnVsbCBpbnRlcnByZXRlciBhcmUgbm90IHBhcnQgb2YgdGhpcyBtb2R1bGUuCgpBZGRpdGlvbmFsIHdvcmsgaW4gdGhpcyBkaXN0cmlidXRpb246IGEgYm91bmRlZCBsYXN0LW5pbmUtYWN0aW9uIGNvbGxlY3Rpb24KYW5kIGRlbGl2ZXJ5IHBsYW5uZXIsIG9ic2VydmF0aW9uIGd1YXJkcyBhbmQgcmVjb3ZlcnksIGEgc2V0dGluZ3MtY29uc3VtaW5nCmZhY3RvcnkgYW5kIGVudHJ5IHBvaW50LCBzdGFuZGFsb25lIGV4YW1wbGVzLCBhbmQgZGV0ZXJtaW5pc3RpYyBwYWNrYWdpbmcuClRoZSBmdWxsIEFwYWNoZSBMaWNlbnNlLCBWZXJzaW9uIDIuMCBpcyBpbmNsdWRlZCBhcyBMSUNFTlNFLnR4dC4KTm8gZW5kb3JzZW1lbnQgYnkgVGhvbWFzIFRzY2hpbmtlbCBvciBLYWdnbGUgaXMgaW1wbGllZC4KCkRhdGEgcHJvdmVuYW5jZSBsaW1pdGF0aW9uOiBUaG9tYXMncyBzb3VyY2UgcmVmZXJzIHRvIHB1YmxpYyByZXBsYXkgZGF0YSBhbmQKYW4gdXBzdHJlYW0gcHJvdmVuYW5jZS5qc29uLiBUaGF0IG9yaWdpbmFsIGVwaXNvZGUtbGV2ZWwgbWFuaWZlc3QsIHJlcGxheSBJRHMKYW5kIGluZGl2aWR1YWwgcmVwbGF5LWF1dGhvciBpZGVudGl0aWVzIHdlcmUgbm90IHN1cHBsaWVkIHdpdGggdGhlIHB1YmxpYwpub3RlYm9vay9vdXRwdXQgdXNlZCBoZXJlLiBObyBuYW1lcyBvciBlcGlzb2RlIGxpbmVhZ2UgaGF2ZSBiZWVuIGludmVudGVkLgpOb3RlYm9vay1sZXZlbCBsaWNlbnNpbmcgZG9lcyBub3QgaW5kZXBlbmRlbnRseSBlc3RhYmxpc2ggdGhlIG1pc3NpbmcgdW5kZXJseWluZwpyZXBsYXktZGF0YSByaWdodHMgY2hhaW4uIFRoZSBwYWNrYWdlIHN1cHBsaWVzIHVzYWJsZSByZWFkYWJsZSByb3V0ZXMsIG5vdCBhCnJlcHJvZHVjaWJsZSByZWNvbnN0cnVjdGlvbiBvZiB0aGVpciBvcmlnaW5hbCBjb2xsZWN0aW9uIG9yIHRyYWluaW5nIHByb2Nlc3MuCgpQYWNrYWdpbmcgbm90ZTogc291cmNlIGlucHV0cyBkZXNjcmliZWQgYXMgYnl0ZS1leGFjdCBhYm92ZSBhcmUKbm9ybWFsaXplZCB0byBVVEYtOC9MRiB0ZXh0IHdpdGggYSBmaW5hbCBuZXdsaW5lIGluIHRoaXMgc3RhbmRhbG9uZQpub3RlYm9vayBwYWNrYWdlLiBSb3V0ZSBKU09OIHZhbHVlcyBhbmQgcGFyZW50IHBvbGljeSBiZWhhdmlvciBhcmUgdW5jaGFuZ2VkLgoKUmVzY3VlIGludGVncmF0aW9uLCBTZXB0ZW1iZXIgMjIsIDIwMjYuCkRtaXRyaWkgR2x1emRvdjogb3BlbmluZyBsaXF1aWRpdHkgcmVwYWlyLCBjb21wbGV0ZSByZWFjdGluZyBldmFsdWF0aW9uIGFuZCBub3RlYm9vayBpbnRlZ3JhdGlvbi4gVGhlIGNvbXBsZXRlIHByZXZpb3VzIFJlc2N1ZSBjb250cm9sbGVyIGFuZCBhbGwgdXBzdHJlYW0gbm90aWNlcyBhcmUgcmV0YWluZWQuIFB1YmxpYyBNYXN0ZXIgSHlicmlkIHdhcyBldmFsdWF0ZWQgYXMgYSBjb21wZXRpbmcgYmFzZTsgaXRzIHJlcGxhY2VtZW50IHdhcyByZWplY3RlZCBhbmQgaXMgbm90IGluY2x1ZGVkIGluIHRoaXMgYXJjaGl2ZS4KCkhlcmQtU2FmZSBjb250cm9sbGVyIHJlZnJlc2gsIFNlcHRlbWJlciAyMiwgMjAyNi4KRG1pdHJpaSBHbHV6ZG92OiBjb21wbGV0ZSBjb250cm9sbGVyIHJlcGxhY2VtZW50IGFuZCByZWFjdGluZyBldmFsdWF0aW9uIGZvciBIZXJkLVNhZmUuIFRoZSB1bmNoYW5nZWQgU2V2ZW4tVHVybiBSZXNjdWUgY29udHJvbGxlciByZXRhaW5zIGFsbCBpbmhlcml0ZWQgbm90aWNlcy4K")
INPUT_ROOT = Path(os.environ.get("KAGGLE_INPUT_DIR", "/kaggle/input"))
OUTPUT_ROOT = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
INPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(data):
    return hashlib.sha256(data).hexdigest()

agents = {}
for archive_path in INPUT_ROOT.rglob("submission.tar.gz"):
    try:
        with tarfile.open(archive_path, "r:gz") as archive:
            member = next((item for item in archive.getmembers() if item.name == "main.py"), None)
            if member is None:
                continue
            agent_source = archive.extractfile(member).read()
        digest = sha256(agent_source)
        if digest in EXPECTED_INPUTS:
            agents[EXPECTED_INPUTS[digest]] = agent_source
    except (OSError, tarfile.TarError):
        continue
missing = {name for name in ("thomas", "herd_safe") if name not in agents}
if missing:
    raise RuntimeError(f"missing verified kernel inputs: {sorted(missing)} under {INPUT_ROOT}")

thomas_blob = base64.b64encode(zlib.compress(agents["thomas"], 9)).decode("ascii")
herd_blob = base64.b64encode(zlib.compress(agents["herd_safe"], 9)).decode("ascii")
candidate = f'''# Locally tested late-game route switch; both controllers stay synchronized.
import base64 as _b64
import copy as _copy
import zlib as _zlib

_THOMAS_NS = {{"__name__": "_late_switch_thomas"}}
_HERD_NS = {{"__name__": "_late_switch_herd_safe"}}
exec(compile(_zlib.decompress(_b64.b64decode({thomas_blob!r})), "thomas.py", "exec"), _THOMAS_NS)
exec(compile(_zlib.decompress(_b64.b64decode({herd_blob!r})), "herd_safe.py", "exec"), _HERD_NS)
_THOMAS_AGENT = _THOMAS_NS["agent"]
_HERD_AGENT = _HERD_NS["agent"]
_SWITCH_STEP = 336

def agent(observation, configuration=None):
    step = int(observation.get("step", 0))
    if step >= _SWITCH_STEP:
        return _HERD_AGENT(_copy.deepcopy(observation), configuration)
    thomas_action = _THOMAS_AGENT(_copy.deepcopy(observation), configuration)
    _HERD_AGENT(_copy.deepcopy(observation), configuration)
    return thomas_action
'''.encode("utf-8")
if sha256(candidate) != EXPECTED_MAIN_SHA256:
    raise RuntimeError(f"candidate main.py hash mismatch: {sha256(candidate)}")
compile(candidate, "main.py", "exec")

archive_path = OUTPUT_ROOT / "submission.tar.gz"
with tarfile.open(archive_path, "w:gz") as archive:
    for filename, data in (("LICENSE.txt", LICENSE_BYTES), ("NOTICE.txt", NOTICE_BYTES), ("main.py", candidate)):
        info = tarfile.TarInfo(filename)
        info.size = len(data)
        info.mode = 0o644
        archive.addfile(info, io.BytesIO(data))
with tarfile.open(archive_path, "r:gz") as archive:
    assert archive.getnames() == ["LICENSE.txt", "NOTICE.txt", "main.py"]
    assert sha256(archive.extractfile("main.py").read()) == EXPECTED_MAIN_SHA256
print("PASS: source hashes", sorted(agents))
print("PASS: wrote submission.tar.gz", archive_path.stat().st_size, "bytes")
print("archive SHA-256:", sha256(archive_path.read_bytes()))
print("main.py SHA-256:", EXPECTED_MAIN_SHA256)
